# 📐 level_flow · trendlines — sloped support and resistance from a chart's own swings

A horizontal level is a line with slope zero. [`trendlines.py`](./trendlines.py) finds lines of **any** slope the way a chart reader draws them, with the rule written down so it can be tested:

1. **Swing points** — a candle whose high (low) is the highest (lowest) of the `pivot_k` candles on each side. A swing is only known `pivot_k` candles later.
2. **Candidate lines** — every pair of same-side swings whose line **price never traded through** in between. Those pairs are exactly the edges of the convex hulls of every sub-window of the chart.
3. **Tests and breaks** — the line is carried forward. A swing back within the tolerance is a **test**, but only a *separate* test if some candle closed `leave_atr` away from the line since the last one. A **close** beyond the line breaks it.
4. **Selection** — at least `min_touches` tests; price must have **traveled along** the line (on average within `max_gap_atr` of it between its first and last test: on an accelerating move every chord between two swings clears the path, so the hull test alone would keep them); a line is only projected as far forward as it was shown to hold (`max_extend` times its tested span); ranked by tests, the time they span and how recent the last one is; one line per idea; only lines within `reach_atr` of the price.

Every distance is in candle ATRs, so one set of knobs reads the same on 15-minute, hourly or daily candles. Lines are straight in **log** price, so every chart here has a log price axis.

| § | figure | question |
|---|---|---|
| 2 | **Fig 1** | how does one chart's handful of lines come out of hundreds of candidates? |
| 3 | **Fig 2** | the lines a trader would have had at `ASOF` — and what price did at them next |
| 4 | **Fig 3** | the same moment on nine other charts |
| 5 | **Fig 4** | one chart through time: lines appear, get tested, break |
| 6 | **Fig 5** | a first honest check: do the lines hold more often than parallel twins drawn at the wrong price? |

**No look-ahead:** candles are stamped at their **close**, and the lines at `ASOF` use only candles finished by then. Everything right of the dotted `ASOF` line is the future the lines are judged on. Set the knobs in §1; `PARAMS` takes any `find_lines` argument.

## 1 · Load a chart

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import plots, strategies
from trading_simple.config import load_settings
from trading_simple.data import load_panel, resample_panel
from trading_simple.db import make_engine

_spec = importlib.util.spec_from_file_location(
    "trendlines", strategies.STRATEGIES_DIR / "level_flow" / "trendlines.py")
tl = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(tl)
settings = load_settings()
engine = make_engine(settings)

SYMBOL = "COIN"
TF = 60              # candle minutes (15, 30, 60, derived from the 1-minute bars) or "1d"
ASOF = "2026-04-15"  # the lines use only candles CLOSED by this moment (New York time)
LOOKBACK = 40        # sessions of swings a line may be anchored on (try 250 for "1d")
FORWARD = 15         # sessions drawn after ASOF, to see what price did at the lines
PARAMS = {}          # any find_lines knob, e.g. {"min_touches": 2, "tol_atr": 0.3}
SHOW_BROKEN = False  # also draw lines that broke BEFORE ASOF (faint, dotted)
GALLERY = ["MU", "COIN", "INTC", "HOOD", "SOFI", "SNOW", "ORCL", "F", "AAOI"]

PER = 1 if TF == "1d" else -(-390 // TF)          # candles per session
KNOBS = {"lookback": LOOKBACK * PER, **PARAMS}
FWD = FORWARD * PER


def load_candles(symbol: str, start, end) -> pd.DataFrame:
    """One symbol's candles at ``TF``, stamped at their CLOSE in New York time.

    Intraday candles are derived from the stored 1-minute bars, session by session, the way a
    chart draws them. Stamping each candle when it CLOSES makes ``ASOF`` honest: the lines see
    a candle only once it is finished (a daily bar is stamped 16:00 on its trade date).
    """
    if TF == "1d":
        p = load_panel(engine, [symbol], "1d", start, end, exchanges=settings.exchanges)
        stamp = p.close.index.tz_convert("UTC").tz_localize(None).normalize()
        stamp = stamp + pd.Timedelta(hours=16)
    else:
        m = load_panel(engine, [symbol], "1m", start, end, session="regular",
                       exchanges=settings.exchanges)
        p = resample_panel(m, TF)
        ny = m.close.index.tz_convert(m.tz).tz_localize(None)
        day = pd.Series(ny.date, index=ny)
        key = [day.to_numpy(), (day.groupby(day).cumcount() // TF).to_numpy()]
        stamp = pd.DatetimeIndex(pd.Series(ny, index=ny).groupby(key).last().to_numpy())
        stamp = stamp + pd.Timedelta(minutes=1)          # the minute after its last bar
        assert len(stamp) == len(p.close), "close stamps out of step with the candles"
    c = pd.DataFrame({"open": p.open_[symbol], "high": p.high[symbol], "low": p.low[symbol],
                      "close": p.close[symbol]})
    c.index = stamp
    return c.dropna()


def window(sessions_back: float, sessions_ahead: float):
    """Calendar dates covering that many sessions around ASOF (weekends and holidays included)."""
    at = pd.Timestamp(ASOF)
    return ((at - pd.Timedelta(days=int(sessions_back * 1.55) + 7)).date(),
            (at + pd.Timedelta(days=int(sessions_ahead * 1.55) + 7)).date())


c = load_candles(SYMBOL, *window(LOOKBACK, FORWARD))
a = tl.asof_position(c, ASOF)
lo, hi = max(0, a - KNOBS["lookback"] + 1), min(a + FWD, len(c) - 1)
print(f"{SYMBOL}: {len(c):,} {TF}{'' if TF == '1d' else 'm'} candles, {c.index[0]:%Y-%m-%d} -> "
      f"{c.index[-1]:%Y-%m-%d}; lines drawn after the candle closing "
      f"{c.index[a]:%a %Y-%m-%d %H:%M}, from {a - lo + 1} candles of history")

In [ ]:
SUP, RES = plots.SERIES[0], plots.SERIES[7]
COLOR = {"support": SUP, "resistance": RES}
UP, DOWN = "#a9c6ba", "#e0b3b0"                    # muted candles: the lines are the subject
OUTCOME = {"held": ("circle", plots._WIN), "broke": ("x", plots._LOSS)}


def names(lines: pd.DataFrame) -> list[str]:
    """R1, R2, ... and S1, S2, ... by score within each side (find_lines' own order)."""
    rank = lines.groupby("side", sort=False).cumcount().add(1).astype(str)
    return (lines["side"].str[0].str.upper() + rank).tolist()


def draw(fig, c, fw, lo, hi, *, row=None, col=None, legend=True, text=True, seen=None):
    """Candles lo..hi, every line solid up to ASOF and dashed after, its tests and outcome.

    ``seen`` is the set of legend entries already shown; pass one set to every panel of a
    multi-panel figure so each kind of mark appears in the legend once.
    """
    rc = {} if row is None else {"row": row, "col": col}
    a, v = fw.attrs["asof"], c.iloc[lo:hi + 1]
    fig.add_trace(go.Candlestick(
        x=np.arange(lo, hi + 1), open=v["open"], high=v["high"], low=v["low"], close=v["close"],
        text=[f"{t:%a %b %d %H:%M}" for t in v.index], hoverinfo="text+y", name="candles",
        increasing_line_color=UP, decreasing_line_color=DOWN, increasing_fillcolor=UP,
        decreasing_fillcolor=DOWN, line={"width": 1}, showlegend=False), **rc)
    seen = set() if seen is None else seen

    def first(kind):
        new = legend and kind not in seen
        seen.add(kind)
        return new

    for i, (r, name) in enumerate(zip(fw.itertuples(), names(fw), strict=True)):
        one, live = fw.iloc[[i]], r.status == "active"
        color, kind = (COLOR[r.side], r.side) if live else (plots._MUTED, "broke before ASOF")
        stop = a if live else int(r.break_pos)
        fig.add_trace(go.Scatter(
            x=[r.x0, stop], y=tl.line_values(one, [r.x0, stop])[0], mode="lines", name=kind,
            legendgroup=kind, showlegend=first(kind),
            line={"color": color, "width": 2.4 if live else 1.2,
                  "dash": "solid" if live else "dot"},
            hovertemplate=f"{name}: {r.side}, {r.touches} tests<extra></extra>"), **rc)
        if live and hi > a:
            fig.add_trace(go.Scatter(
                x=[a, hi], y=tl.line_values(one, [a, hi])[0], mode="lines", legendgroup=kind,
                showlegend=False, hoverinfo="skip",
                line={"color": color, "width": 1.4, "dash": "dash"}), **rc)
        tp = np.asarray(r.touch_pos)
        fig.add_trace(go.Scatter(
            x=tp, y=tl.line_values(one, tp)[0], mode="markers", name="a test of the line",
            legendgroup="test", showlegend=first("test"),
            marker={"size": 8, "color": color, "line": {"color": "white", "width": 1}},
            hovertemplate=f"{name} test %{{y:.2f}}<extra></extra>"), **rc)
        if text and live:                              # the name where the line starts
            fig.add_trace(go.Scatter(
                x=[r.x0], y=tl.line_values(one, [r.x0])[0], mode="text", text=[name],
                textposition="top center" if r.side == "resistance" else "bottom center",
                textfont={"size": 12, "color": color}, showlegend=False, hoverinfo="skip"), **rc)
        if live and getattr(r, "outcome", None) in OUTCOME:
            sym, oc = OUTCOME[r.outcome]
            fig.add_trace(go.Scatter(
                x=[r.test_pos], y=tl.line_values(one, [r.test_pos])[0], mode="markers",
                name=f"price came back to it: {r.outcome}", legendgroup=r.outcome,
                showlegend=first(r.outcome),
                marker={"symbol": sym, "size": 13, "color": oc,
                        "line": {"color": oc, "width": 2.5}},
                hovertemplate=f"{name} tested %{{x}}: {r.outcome}<extra></extra>"), **rc)
    # Shapes go on AFTER the traces of their subplot, or plotly drops them silently.
    fig.add_vline(x=a + 0.5, line={"color": plots._INK, "width": 1, "dash": "dot"}, **rc)
    # A LOG price axis: the lines are straight in log price, so on a linear axis a two-point
    # segment would be a chord of the true line (visibly so on a large daily move).
    lo_y, hi_y = np.log10(v["low"].min()), np.log10(v["high"].max())
    pad = 0.07 * (hi_y - lo_y)                            # room for the line names
    fig.update_yaxes(type="log", range=[lo_y - pad, hi_y + pad], **log_ticks(lo_y, hi_y), **rc)
    ticks, labels = date_ticks(c, lo, hi)
    fig.update_xaxes(tickvals=ticks, ticktext=labels, range=[lo - 1, hi + 1],
                     rangeslider_visible=False, **rc)


def log_ticks(lo_y: float, hi_y: float) -> dict:
    """Full-number 1-2-5 ticks on a log axis spanning ~5x or more (plotly labels its own minor
    ticks "2" and "5"); a narrower range keeps plotly's finer automatic ticks."""
    if hi_y - lo_y <= 0.7:
        return {}
    vals = [m * 10.0 ** k for k in range(int(np.floor(lo_y)), int(np.ceil(hi_y)) + 1)
            for m in (1, 2, 5) if lo_y <= np.log10(m * 10.0 ** k) <= hi_y]
    return {"tickvals": vals, "ticktext": [f"{v:g}" for v in vals]}


def date_ticks(c, lo, hi, n=7):
    """Tick positions at the first candle of each session (each month on daily candles)."""
    d = pd.Series(c.index[lo:hi + 1], index=np.arange(lo, hi + 1))
    key = d.dt.to_period("M") if TF == "1d" else d.dt.date
    first = d[key != key.shift()]
    step = max(1, -(-len(first) // n))
    return first.index[::step].tolist(), [f"{t:%b %d}" for t in first.iloc[::step]]


def legend_right(fig, width=190):
    fig.update_layout(legend={"orientation": "v", "yanchor": "top", "y": 1.0, "xanchor": "left",
                              "x": 1.01}, margin={"r": width})

## 2 · How the lines are found (Fig 1)

The chart up to `ASOF` only. **Grey triangles** are the confirmed swings. The **faint lines** are every pair of same-side swings whose line price never traded through in between, i.e. every line a ruler could have been laid on without cutting a candle, each drawn until price closed through it. The **bold lines** are the few that survive: separate tests, a small tolerance, one line per idea, near the current price. Hover a bold line for its test count.

What to look for: the kept lines should be the ones your own eye picks out of the faint cloud, not arbitrary members of it; and each should rest on swings that are clearly separate visits to the line, not one wobble counted several times.

In [ ]:
sw = tl.swings(c.iloc[:a + 1], KNOBS.get("pivot_k", 3))
sw = sw[(sw["pos"] >= lo) & (sw["known"] <= a)]
cand = tl.find_lines(c, ASOF, candidates=True, **KNOBS)
lines = tl.find_lines(c, ASOF, **KNOBS)

fig = go.Figure()
for side in tl.SIDES:                                # the cloud first, so it sits behind
    cs = cand[cand["side"] == side]
    x0, y0, sl = (cs[k].to_numpy() for k in ("x0", "y0", "slope"))
    x1 = np.where(cs["break_pos"].isna(), a, cs["break_pos"]).astype(int)  # alive until here
    gap = np.full(len(cs), np.nan)                   # NaN ends each segment
    fig.add_trace(go.Scatter(
        x=np.c_[x0, x1, gap].ravel(), y=np.exp(np.c_[y0, y0 + sl * (x1 - x0), gap]).ravel(),
        mode="lines", opacity=0.22, hoverinfo="skip", line={"color": COLOR[side], "width": 0.8},
        name=f"{len(cs)} {side} candidates"))
draw(fig, c, lines, lo, a)
for side, mark in (("resistance", "triangle-down"), ("support", "triangle-up")):
    s = sw[sw["side"] == side]
    fig.add_trace(go.Scatter(x=s["pos"], y=np.exp(s["y"]), mode="markers",
                             name=f"swing {'high' if side == 'resistance' else 'low'}",
                             marker={"symbol": mark, "size": 6, "color": plots._MUTED},
                             hoverinfo="skip"))
plots._base(fig, title=f"<b>Fig 1</b> · {SYMBOL} · {len(sw)} swings → {len(cand)} lines price "
                       f"never crossed between two of them → {len(lines)} kept", height=520)
legend_right(fig, 210)
fig.show()

## 3 · The lines at `ASOF`, and what price did at them next (Fig 2)

Each line is **solid** over the history it was built from and **dashed** into the future it is judged on. Dots are its tests; `R1`/`S1` label the best resistance and support. When price first came back to a line after `ASOF`, the marker says what followed: **green circle = held** (price retreated a full candle ATR before any close through the line), **red cross = broke** (a close through it came first). The table below lists every line with its slope in % per session, its distance from the price in ATRs, and its outcome.

What to look for: a line that "held" only because price never got near it is not in the table as held. It is `untested`. A chart whose lines are all tested and all broke is telling you something about the name, not the drawing.

In [ ]:
lines = tl.find_lines(c, ASOF, keep_broken=SHOW_BROKEN, **KNOBS)
fw = tl.forward_tests(lines, c, FWD)
fig = go.Figure()
draw(fig, c, fw, lo, hi)
fig.add_annotation(x=a + 0.5, y=1.0, yref="paper", showarrow=False, xanchor="left",
                   yanchor="bottom", text="  lines drawn here → judged on what follows",
                   font={"size": 11, "color": plots._MUTED})
live = fw[fw["status"] == "active"]
tally = live["outcome"].value_counts().reindex(["held", "broke", "open", "untested"], fill_value=0)
what = ", ".join(f"{v} {k}" for k, v in tally.items() if v) or "no line clears the rules here"
plots._base(fig, title=f"<b>Fig 2</b> · {SYMBOL} · {len(live)} line{'' if len(live) == 1 else 's'} "
                       f"at {c.index[a]:%b %d %H:%M}: {what}", height=560)
legend_right(fig)
fig.show()


def line_table(fw: pd.DataFrame, c: pd.DataFrame) -> pd.DataFrame:
    """One row per line: its tests, geometry, reading at ASOF and forward outcome."""
    when = lambda p: "—" if pd.isna(p) else f"{c.index[int(p)]:%b %d %H:%M}"  # noqa: E731
    return pd.DataFrame({
        "line": names(fw), "side": fw["side"], "status": fw["status"], "tests": fw["touches"],
        "first test": [when(p) for p in fw["x0"]], "last test": [when(p) for p in fw["last_touch"]],
        "span (sessions)": (fw["last_touch"] - fw["x0"]) / PER,
        "slope (%/session)": 100 * fw["slope"] * PER, "gap (ATR)": fw["gap_atr"],
        "level at ASOF": fw["level"],
        "distance (ATR)": fw["dist_atr"], "tested": [when(p) for p in fw["test_pos"]],
        "outcome": fw["outcome"]}).set_index("line")


display(line_table(fw, c).style.format({
    "span (sessions)": "{:.1f}", "slope (%/session)": "{:+.2f}", "gap (ATR)": "{:.1f}",
    "level at ASOF": "{:.2f}",
    "distance (ATR)": "{:+.1f}"}))

## 4 · The same moment on other charts (Fig 3)

`GALLERY` at the same `ASOF` and knobs, one panel per name, drawn exactly like Fig 2. Panel titles count the lines and their outcomes. This is the view for your real question — *which names are in level territory right now?* — by eye first: a panel with several multi-test lines that held is a different animal from one with no lines at all or lines that all broke.

In [ ]:
CANDLES = {}                                          # §6 reuses these loads
PLACEBO_SESSIONS = 180                                # §6: how far back its ASOF grid reaches
for sym in GALLERY:
    CANDLES[sym] = load_candles(sym, *window(PLACEBO_SESSIONS + LOOKBACK, FORWARD))
panels = []
for sym in GALLERY:
    cg = CANDLES[sym]
    ag = tl.asof_position(cg, ASOF)
    fg = tl.forward_tests(tl.find_lines(cg, ASOF, **KNOBS), cg, FWD)
    lo_g = max(0, ag - KNOBS["lookback"] + 1)
    panels.append((sym, cg, fg, lo_g, min(ag + FWD, len(cg) - 1)))
titles = []
for sym, _, fg, _, _ in panels:
    n = fg["outcome"].value_counts()
    titles.append(f"<b>{sym}</b> · {len(fg)} line{'' if len(fg) == 1 else 's'} · "
                  f"{n.get('held', 0)} held, {n.get('broke', 0)} broke")
cols = 3
rows = -(-len(panels) // cols)
fig = make_subplots(rows=rows, cols=cols, subplot_titles=titles, horizontal_spacing=0.045,
                    vertical_spacing=0.09)
seen = set()
for k, (_, cg, fg, lo_g, hi_g) in enumerate(panels):
    draw(fig, cg, fg, lo_g, hi_g, row=k // cols + 1, col=k % cols + 1, text=False, seen=seen)
plots._base(fig, title=f"<b>Fig 3</b> · the lines at {pd.Timestamp(ASOF):%Y-%m-%d}, "
                       f"{TF}{'' if TF == '1d' else 'm'} candles, on {len(panels)} charts",
            height=330 * rows)
fig.update_annotations(font_size=12)
fig.update_xaxes(tickfont={"size": 9})
fig.update_yaxes(tickfont={"size": 9})
legend_right(fig, 200)
fig.show()

## 5 · One chart through time (Fig 4)

`SYMBOL` at six moments `STEP` sessions apart, ending at `ASOF`, each panel with its own history and its own future. Lines should be *stable*: the same line should appear panel after panel until price breaks it, not flicker in and out as a new swing arrives. A method that redraws the chart every day cannot be traded, whatever its hit rate.

In [ ]:
STEP = 5                                              # sessions between panels
cw = load_candles(SYMBOL, *window(LOOKBACK + 5 * STEP, FORWARD))
aw = tl.asof_position(cw, ASOF)
moments = [aw - k * STEP * PER for k in range(5, -1, -1)]
fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.045, vertical_spacing=0.12,
                    subplot_titles=[f"as of {cw.index[m]:%b %d %H:%M}" for m in moments])
seen = set()
for k, m in enumerate(moments):
    fk = tl.forward_tests(tl.find_lines(cw, m, **KNOBS), cw, FWD)
    lo_k = max(0, m - KNOBS["lookback"] + 1)
    draw(fig, cw, fk, lo_k, min(m + FWD, len(cw) - 1), row=k // 3 + 1, col=k % 3 + 1, seen=seen)
plots._base(fig, title=f"<b>Fig 4</b> · {SYMBOL} · the lines every {STEP} sessions, each judged "
                       f"on the {FORWARD} sessions after it", height=700)
fig.update_annotations(font_size=12)
fig.update_xaxes(tickfont={"size": 9})
fig.update_yaxes(tickfont={"size": 9})
legend_right(fig, 200)
fig.show()

## 6 · A first honest check: do the lines hold better than their parallel twins? (Fig 5)

A pretty chart proves nothing. Random lines also "hold" much of the time: over a short window, price mean-reverts. The placebo for a line is its **parallel twin**: same slope, same start, moved 0.5–2 candle ATRs up or down, so it keeps the geometry but loses the swings that built it. Twins that land on the wrong side of the price are dropped.

Every `GALLERY` chart is sampled every `FORWARD` sessions over the last `PLACEBO_SESSIONS`, so no two samples of a chart share a future. Each line and each twin is forward-tested the same way as Fig 2. The only fair comparison is **held, given tested**; how often a line is tested at all depends mostly on its distance from price.

Read it as a smoke test, not a verdict: a few hundred lines on nine names, in-sample for the knobs, and an unbroken line is sampled again at the next moment, so the ± bars are too narrow. The real test is the pre-registered, many-name levelness study this notebook is the drawing tool for.

In [ ]:
rng = np.random.default_rng(7)
records = []
for sym, cg in CANDLES.items():
    last = tl.asof_position(cg, ASOF)
    for m in range(last, KNOBS["lookback"], -FWD):   # forward windows never overlap
        lines_m = tl.find_lines(cg, m, **KNOBS)
        if lines_m.empty:
            continue
        shift = rng.choice([-1.0, 1.0], len(lines_m)) * rng.uniform(0.5, 2.0, len(lines_m))
        twins = tl.shifted(lines_m, shift)
        side = np.where(twins["side"] == "resistance", 1.0, -1.0)
        tol_atr = KNOBS.get("tol_atr", 0.25)
        twins = twins[side * twins["dist_atr"] > tol_atr]       # still on its side of the price
        for kind, lm in (("line", lines_m), ("parallel twin", twins)):
            if len(lm):
                f = tl.forward_tests(lm, cg, FWD)
                records.append(f[["side", "touches", "outcome"]].assign(kind=kind, symbol=sym))
ev = pd.concat(records, ignore_index=True)
ev["tested"] = ev["outcome"].isin(["held", "broke", "open"])
ev["held"] = ev["outcome"] == "held"
res = ev[ev["outcome"].isin(["held", "broke"])]
summary = res.groupby(["side", "kind"]).agg(resolved=("held", "size"), held=("held", "mean"))
summary["± se"] = np.sqrt(summary["held"] * (1 - summary["held"]) / summary["resolved"])
summary.insert(0, "lines", ev.groupby(["side", "kind"]).size())
summary.insert(1, "tested", ev.groupby(["side", "kind"])["tested"].mean())
display(summary.style.format({"tested": "{:.0%}", "held": "{:.0%}", "± se": "{:.0%}"}))

fig = go.Figure()
for kind, color in (("line", plots._INK), ("parallel twin", plots._MUTED)):
    s = summary.xs(kind, level="kind")
    fig.add_trace(go.Bar(x=s.index, y=s["held"], name=kind, marker_color=color,
                         error_y={"type": "data", "array": 1.96 * s["± se"], "color": "#555"},
                         text=[f"{h:.0%}<br>n={n}" for h, n in zip(s["held"], s["resolved"],
                                                                     strict=True)],
                         textposition="inside", insidetextanchor="start",
                         textfont={"color": "white", "size": 11}))
fig.add_hline(y=0.5, line={"color": plots._MUTED, "dash": "dot", "width": 1})
plots._base(fig, title=f"<b>Fig 5</b> · held when tested: {len(res):,} resolved tests of lines "
                       f"and twins on {len(CANDLES)} charts (bars ±1.96 se)", height=420)
fig.update_layout(barmode="group", bargap=0.35)
fig.update_yaxes(tickformat=".0%", range=[0, 1], title_text="held, given tested")
fig.show()

## Read it honestly

* The rules are the chart reader's, written down; they are not evidence. Figs 1–4 show that the method draws the lines a person would, and where it does not. They cannot show that those lines matter.
* **Fig 5 is the only number here.** A line family that does not beat its parallel twins on *held, given tested* is decoration, however clean it looks. That is the same test the horizontal levels failed (the placebo in the level_flow README).
* Tune by eye with `PARAMS`: `min_touches: 2` shows young lines, `tol_atr` loosens what counts as a touch, `leave_atr` how far price must leave between tests, `reach_atr` how far from the price a line may be, `max_extend: None` keeps lines long after their last test, and `max_gap_atr: None` shows the chords of accelerating moves (and old extremes retested after a long excursion, which are a horizontal level's job, not a trendline's). The table's `gap (ATR)` is how far price stood from each line on average.
* `TF` changes the question as much as the answer: hourly lines are for days, daily lines for months. A line that exists on both is a stronger claim than either alone.